# Phase 23 — economic interpretation of multiscale regimes

This notebook builds state cards for the model selected by the frozen phase-22 validation rule. Economic interpretation means describing current hierarchy, persistence, activity, and later diagnostic behaviour. It does not create entries, exits, position direction, or a backtest.

In [1]:
import os

# Reproducible HMM fits (M15): fix the BLAS/OpenMP and PyTorch thread counts before the numerical
# libraries load. A concurrent or differently threaded run can change floating-point reduction order
# and therefore the EM path. Restart the kernel after changing HMM_THREADS.
HMM_THREADS = 1
for _thread_variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_thread_variable, str(HMM_THREADS))

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from dc_basics.data import resolve_data_root
from dc_multiscale_research import (
    FeatureTier,
    ModelFamily,
    ObservationClock,
    REGIME_ORDER,
    add_future_diagnostics,
    build_multiscale_state,
    load_close,
    plot_price_states,
    prepare_data,
    ratio_ladder,
    select_hmm_candidates,
    semantic_state_name,
    transition_matrix,
)

torch.set_num_threads(HMM_THREADS)

DATA_ROOT = resolve_data_root()
SYMBOL = "GBPUSD.r"
THRESHOLDS = ratio_ladder(0.002, 1.5, 5)
CLOCK = ObservationClock.UNION_EVENT
FEATURE_TIER = FeatureTier.DYNAMIC
STATE_COUNTS = (3, 4, 5)
HMM_FAMILIES = (ModelFamily.NORMAL_HMM, ModelFamily.STUDENTT_HMM)
RANDOM_STATE = 17
STABILITY_SEEDS = (29, 47)
MAX_ITER = 50
N_RESTARTS = 3
MINIMUM_VALIDATION_OCCUPANCY = 0.02
MINIMUM_RESTART_ARI = 0.80
MIN_EMISSION_VARIANCE = 1e-2
FUTURE_HORIZONS = (6, 24, 96)
LAST_N = 8_000

## Reproduce the frozen validation selection

No economic profile participates in model selection. The family and state count are chosen using validation likelihood only after prefix causality, occupancy, and restart-stability eligibility gates pass. The direct structural fields used to interpret a state were not supplied to the continuous emission model.

In [2]:
prices = load_close(DATA_ROOT, SYMBOL)
_, global_state, _ = build_multiscale_state(prices, THRESHOLDS, include_scale_state=False)
prepared = prepare_data(global_state, CLOCK, FEATURE_TIER)

selection = select_hmm_candidates(
    prepared,
    HMM_FAMILIES,
    STATE_COUNTS,
    random_state=RANDOM_STATE,
    stability_seeds=STABILITY_SEEDS,
    max_iter=MAX_ITER,
    n_restarts=N_RESTARTS,
    minimum_validation_occupancy=MINIMUM_VALIDATION_OCCUPANCY,
    minimum_restart_ari=MINIMUM_RESTART_ARI,
    min_emission_variance=MIN_EMISSION_VARIANCE,
)
selection_table = selection.table
selected_fit = selection.selected_fit
if selected_fit is None:
    raise RuntimeError("No eligible HMM candidate; inspect selection_table.")
display(selection_table.set_index(["family", "n_states"]))
display(selection.stability.set_index(["family", "n_states", "seed"]))
print(f"Interpretation model: {selected_fit.family.value}, K={selected_fit.n_states}")

clock     tier  feature_count  \
family       n_states                                             
normal_hmm   3         union_event_time  dynamic             21   
             4         union_event_time  dynamic             21   
             5         union_event_time  dynamic             21   
studentt_hmm 3         union_event_time  dynamic             21   
             4         union_event_time  dynamic             21   
             5         union_event_time  dynamic             21   

                       removed_feature_count  prepared_rows_dropped error  \
family       n_states                                                       
normal_hmm   3                             1                      0  None   
             4                             1                      0  None   
             5                             1                      0  None   
studentt_hmm 3                             1                      0  None   
             4                             1                      0  None   
             5                             1                      0  None   

                       validation_log_likelihood_per_row  \
family       n_states                                      
normal_hmm   3                                -22.590537   
             4                                -19.738444   
             5                                -18.732857   
studentt_hmm 3                                -22.896883   
             4                                -21.071885   
             5                                -20.080213   

                       minimum_state_occupancy  switch_rate  median_dwell  \
family       n_states                                                       
normal_hmm   3                        0.292111     0.947705           1.0   
             4                        0.156716     0.896478           1.0   
             5                        0.126866     0.935966           1.0   
studentt_hmm 3                        0.239872     0.788687           1.0   
             4                        0.124733     0.872999           1.0   
             5                        0.131130     0.899680           1.0   

                       ...  baseline_nmi  semantic_mapping_coverage  \
family       n_states  ...                                            
normal_hmm   3         ...      0.330369                   1.000000   
             4         ...      0.511658                   1.000000   
             5         ...      0.266190                   0.786780   
studentt_hmm 3         ...      0.201646                   1.000000   
             4         ...      0.387654                   1.000000   
             5         ...      0.439902                   0.760128   

                       baseline_mapped_agreement  minimum_fitted_variance  \
family       n_states                                                       
normal_hmm   3                          0.479744                     0.01   
             4                          0.651386                     0.01   
             5                          0.510840                     0.01   
studentt_hmm 3                          0.473348                     0.01   
             4                          0.603412                     0.01   
             5                          0.807854                     0.01   

                       variance_floor_hit_count  variance_floor_hit_share  \
family       n_states                                                       
normal_hmm   3                                3                  0.047619   
             4                                5                  0.059524   
             5                               13                  0.123810   
studentt_hmm 3                                5                  0.079365   
             4                                5                  0.059524   
             5                                6          

error  validation_ari_vs_primary  \
family       n_states seed                                    
normal_hmm   3        29    None                   0.769205   
                      47    None                   0.668122   
             4        29    None                   1.000000   
                      47    None                   1.000000   
             5        29    None                   0.545847   
                      47    None                   0.783602   
studentt_hmm 3        29    None                   1.000000   
                      47    None                   0.724244   
             4        29    None                   1.000000   
                      47    None                   1.000000   
             5        29    None                   0.664581   
                      47    None                   0.757572   

                            validation_log_likelihood_per_row  \
family       n_states seed                                      
normal_hmm   3        29                           -21.578788   
                      47                           -22.143909   
             4        29                           -19.738497   
                      47                           -19.738903   
             5        29                           -18.453852   
                      47                           -18.806643   
studentt_hmm 3        29                           -22.894412   
                      47                           -23.661815   
             4        29                           -21.071653   
                      47                           -21.071653   
             5        29                           -20.063935   
                      47                           -20.319191   

                            prefix_invariant  
family       n_states seed                    
normal_hmm   3        29                True  
                      47                True  
             4        29                True  
                      47                True  
             5        29                True  
                      47                True  
studentt_hmm 3        29                True  
                      47                True  
             4        29                True  
                      47                True  
             5        29                True  
                      47                True

Interpretation model: normal_hmm, K=4


## Construct one causal state path

Raw model states are retained. Training-only semantic names are annotations that explain strongest overlap with the direct taxonomy; they do not redefine the fitted states.

In [3]:
state_parts = []
for split_name in ("train", "validation", "test"):
    part = prepared.frames[split_name].copy()
    part["partition"] = split_name
    part["model_state"] = selected_fit.states[split_name]
    part["model_state_label"] = [f"state_{state}" for state in selected_fit.states[split_name]]
    part["training_overlap_annotation"] = [
        semantic_state_name(code) for code in selected_fit.mapped_regimes[split_name]
    ]
    part["filtered_confidence"] = selected_fit.probabilities[split_name].max(axis=1)
    for state in range(selected_fit.n_states):
        part[f"filtered_p_state_{state}"] = selected_fit.probabilities[split_name][:, state]
    state_parts.append(part)
state_path = pd.concat(state_parts).sort_index()
if not state_path["feature_information_time"].eq(state_path["time"]).all():
    raise AssertionError("A state-path feature timestamp differs from its observation time.")
display(state_path.tail(10))

,time,feature_information_time,bar_position,price,threshold_count,configuration_code,configuration,broad_agreement,coherent_frontier,direction_boundaries,...,log_completed_r_04,partition,model_state,model_state_label,training_overlap_annotation,filtered_confidence,filtered_p_state_0,filtered_p_state_1,filtered_p_state_2,filtered_p_state_3
time,,,,,,,,,,,,,,,,,,,,,
2026-11-05 14:15:00+02:00,2026-11-05 14:15:00+02:00,2026-11-05 14:15:00+02:00,258005,1.34774,5,29,up/down/up/up/up,False,False,2,...,-11.589347,test,1,state_1,fragmented,1.0,2.016896e-41,1.000000e+00,1.148377e-131,3.147410e-135
2026-11-05 21:15:00+02:00,2026-11-05 21:15:00+02:00,2026-11-05 21:15:00+02:00,258089,1.34549,5,28,down/down/up/up/up,False,True,1,...,-11.589347,test,1,state_1,fragmented,1.0,5.944223e-51,1.000000e+00,2.810699e-160,1.677051e-98
2026-11-06 05:15:00+02:00,2026-11-06 05:15:00+02:00,2026-11-06 05:15:00+02:00,258185,1.34374,5,24,down/down/down/up/up,False,True,1,...,-11.589347,test,2,state_2,deep_counter,1.0,1.394343e-41,1.764655e-159,1.000000e+00,9.301974e-137
2026-11-06 09:00:00+02:00,2026-11-06 09:00:00+02:00,2026-11-06 09:00:00+02:00,258230,1.34022,5,16,down/down/down/down/up,False,True,1,...,-11.589347,test,2,state_2,deep_counter,1.0,2.644638e-37,1.755834e-228,1.000000e+00,1.154504e-204
2026-11-06 15:25:00+02:00,2026-11-06 15:25:00+02:00,2026-11-06 15:25:00+02:00,258307,1.33636,5,0,down/down/down/down/down,True,True,0,...,-11.386649,test,0,state_0,broad_coherence,1.0,1.000000e+00,0.000000e+00,1.176814e-36,0.000000e+00
2026-11-06 17:20:00+02:00,2026-11-06 17:20:00+02:00,2026-11-06 17:20:00+02:00,258330,1.33768,5,1,up/down/down/down/down,False,True,1,...,-11.386649,test,1,state_1,fragmented,1.0,9.065586e-95,1.000000e+00,1.332663e-177,0.000000e+00
2026-11-06 21:45:00+02:00,2026-11-06 21:45:00+02:00,2026-11-06 21:45:00+02:00,258383,1.33490,5,0,down/down/down/down/down,True,True,0,...,-11.386649,test,0,state_0,broad_coherence,1.0,1.000000e+00,2.274144e-28,3.484399e-31,3.419629e-306
2026-11-09 03:15:00+02:00,2026-11-09 03:15:00+02:00,2026-11-09 03:15:00+02:00,258449,1.33713,5,1,up/down/down/down/down,False,True,1,...,-11.386649,test,1,state_1,fragmented,1.0,3.825535e-79,1.000000e+00,2.197885e-155,0.000000e+00
2026-11-09 08:10:00+02:00,2026-11-09 08:10:00+02:00,2026-11-09 08:10:00+02:00,258508,1.33851,5,3,up/up/down/down/down,False,True,1,...,-11.386649,test,3,state_3,shallow_counter,1.0,7.309037e-47,1.849706e-127,1.259996e-137,1.000000e+00


## State cards: current economic structure

These fields answer what the state means **now**: direction balance, hierarchy coherence, scale reach, live reversal pressure, event activity, confidence, and direct-regime overlap.

`counter_depth_fraction` exists only for rows with zero or one boundary; fragmented rows have no single counter depth, so that column's `count` is lower than the others in states that contain fragmented rows. Discrete descriptors such as `fragmentation` and `signed_mode_balance` were not model inputs; they describe each state after fitting.

In [4]:
current_fields = [
    "fragmentation",
    "counter_depth_fraction",
    "signed_mode_balance",
    "mean_mode_age_bars",
    "mean_reversal_progress",
    "max_reversal_progress",
    "hierarchy_dcc_rate",
    "filtered_confidence",
]
state_cards = state_path.groupby("model_state", observed=True)[current_fields].agg(
    ["count", "median", "mean"]
)
baseline_overlap = pd.crosstab(
    state_path["model_state"], state_path["structural_regime"], normalize="index"
).reindex(columns=REGIME_ORDER, fill_value=0.0)
direction_overlap = pd.crosstab(
    state_path["model_state"], state_path["outer_mode"], normalize="index"
)
display(state_cards)
display(baseline_overlap.style.format("{:.2%}"))
display(direction_overlap.style.format("{:.2%}"))

fragmentation                  counter_depth_fraction         \
                    count median      mean                  count median   
model_state                                                                
0                     790   0.00  0.000000                    790    0.0   
1                    2238   0.50  0.438896                    959    0.2   
2                     924   0.25  0.291937                    767    0.6   
3                    1110   0.25  0.382432                    608    0.4   

                      signed_mode_balance                  mean_mode_age_bars  \
                 mean               count median      mean              count   
model_state                                                                     
0            0.000000                 790    1.0  0.048101                790   
1            0.267987                2238    0.2  0.015371               2238   
2            0.692568                 924   -0.2 -0.013853                924   
3            0.442434                1110    0.2  0.008649               1110   

             ... mean_reversal_progress max_reversal_progress            \
             ...                   mean                 count    median   
model_state  ...                                                          
0            ...               0.038827                   790  0.072062   
1            ...               0.371460                  2238  0.721063   
2            ...               0.202995                   924  0.697947   
3            ...               0.300027                  1110  0.701827   

                      hierarchy_dcc_rate                  filtered_confidence  \
                 mean              count median      mean               count   
model_state                                                                     
0            0.096361                790  0.005  0.005442                 790   
1            0.740487               2238  0.005  0.005322                2238   
2            0.728484                924  0.005  0.005946                 924   
3            0.719828               1110  0.005  0.005706                1110   

                              
            median      mean  
model_state                   
0              1.0  0.998664  
1              1.0  0.998995  
2              1.0  0.994387  
3              1.0  0.997482  

[4 rows x 24 columns]

structural_regime,broad_coherence,shallow_counter,deep_counter,fragmented
model_state,,,,
0,100.00%,0.00%,0.00%,0.00%
1,0.18%,38.20%,4.47%,57.15%
2,0.22%,0.00%,82.79%,16.99%
3,0.18%,46.22%,8.38%,45.23%


outer_mode,down,up
model_state,,
0,47.59%,52.41%
1,46.51%,53.49%
2,45.56%,54.44%
3,46.49%,53.51%


## Persistence and transition meaning

Transitions are estimated on the held-out event-time path. They describe how structural events move between inferred states; they are not price-direction forecasts.

In [5]:
test_states = selected_fit.states["test"]
test_transition = pd.DataFrame(
    transition_matrix(test_states, selected_fit.n_states),
    index=[f"state_{state}" for state in range(selected_fit.n_states)],
    columns=[f"state_{state}" for state in range(selected_fit.n_states)],
)
display(test_transition.style.format("{:.3f}"))

semantic_map = pd.Series({
    state: semantic_state_name(regime_code)
    for state, regime_code in selected_fit.semantic_map.items()
}, name="training_overlap_annotation")
display(semantic_map.to_frame())

,state_0,state_1,state_2,state_3
state_0,0.008,0.967,0.016,0.008
state_1,0.193,0.196,0.135,0.476
state_2,0.210,0.586,0.167,0.037
state_3,0.080,0.481,0.439,0.000


,training_overlap_annotation
0,broad_coherence
1,fragmented
2,deep_counter
3,shallow_counter


## Later diagnostic behaviour

Forward absolute movement, realized volatility, and multiscale DCC count help name a regime economically. They remain hindsight outcomes and never enter the causal state assignment. Results are shown separately for the chronological test partition.

In [6]:
diagnostics = add_future_diagnostics(global_state, FUTURE_HORIZONS)
future_columns = []
for horizon in FUTURE_HORIZONS:
    future_columns.extend([
        f"future_return_{horizon}",
        f"future_abs_return_{horizon}",
        f"future_realized_vol_{horizon}",
        f"future_dcc_count_{horizon}",
    ])
test_cards = state_path.loc[state_path["partition"].eq("test")].join(
    diagnostics[future_columns], how="left"
)
future_profile = test_cards.groupby("model_state", observed=True)[future_columns].agg(["median", "mean"])
display(future_profile)

directional_check = test_cards.groupby("model_state", observed=True)[
    [f"future_return_{horizon}" for horizon in FUTURE_HORIZONS]
].mean()
display(directional_check.style.format("{:.4%}"))
print("Directional means are diagnostic and do not authorize a directional regime or strategy interpretation.")

future_return_6           future_abs_return_6            \
                     median      mean              median      mean   
model_state                                                           
0                 -0.000038 -0.000011            0.000419  0.000636   
1                  0.000057  0.000044            0.000354  0.000494   
2                 -0.000015 -0.000023            0.000464  0.000646   
3                  0.000052  0.000003            0.000492  0.000621   

            future_realized_vol_6           future_dcc_count_6            \
                           median      mean             median      mean   
model_state                                                                
0                        0.000689  0.000813                0.0  0.073171   
1                        0.000543  0.000630                0.0  0.153439   
2                        0.000733  0.000842                0.0  0.098160   
3                        0.000646  0.000772                0.0  0.144385   

            future_return_24            ... future_dcc_count_24            \
                      median      mean  ...              median      mean   
model_state                             ...                                 
0                   0.000263  0.000303  ...                 0.0  0.447154   
1                   0.000015  0.000064  ...                 0.0  0.708995   
2                  -0.000136  0.000048  ...                 0.0  0.607362   
3                   0.000142  0.000096  ...                 0.0  0.679144   

            future_return_96           future_abs_return_96            \
                      median      mean               median      mean   
model_state                                                             
0                   0.000375  0.000330             0.001562  0.002230   
1                  -0.000055  0.000207             0.001411  0.001941   
2                  -0.000222  0.000123             0.001567  0.002106   
3                   0.000134  0.000098             0.001473  0.002031   

            future_realized_vol_96           future_dcc_count_96            
                            median      mean              median      mean  
model_state                                                                 
0                         0.002641  0.002890                 1.0  2.081301  
1                         0.002416  0.002665                 2.0  2.404762  
2                         0.002450  0.002726                 1.0  2.018519  
3                         0.002577  0.002801                 2.0  2.413978  

[4 rows x 24 columns]

,future_return_6,future_return_24,future_return_96
model_state,,,
0,-0.0011%,0.0303%,0.0330%
1,0.0044%,0.0064%,0.0207%
2,-0.0023%,0.0048%,0.0123%
3,0.0003%,0.0096%,0.0098%


Directional means are diagnostic and do not authorize a directional regime or strategy interpretation.


## Actual price path and filtered confidence

In [7]:
probability_columns = [f"filtered_p_state_{state}" for state in range(selected_fit.n_states)]
plot_price_states(
    state_path,
    "model_state_label",
    f"{SYMBOL}: economically inspected raw causal filtered states",
    probability_columns=probability_columns,
    last_n=LAST_N,
).show()
plot_price_states(
    state_path,
    "directed_regime",
    f"{SYMBOL}: direction-aware direct structural comparator on the same observations",
    last_n=LAST_N,
).show()

# Phase 23 completion gate

For each inferred state, be able to state:

1. Its raw state ID and, separately, its training-only overlap annotation.
2. Its current hierarchy shape and direction balance.
3. Whether it represents agreement, a coherent frontier, or fragmentation.
4. Its scale ages, reversal pressure, and DCC activity.
5. Its filtered confidence, persistence, restart stability, and any variance-floor warning.
6. How it transitions to other states.
7. Which later activity/volatility differences are diagnostic rather than causal inputs.
8. Whether it adds a stable distinction beyond both direct baselines.
9. Why none of this yet specifies a position direction, entry, exit, or expected net return.